# Hawk MiniMax H3 Director · API on Colab

Runs ComfyUI with the Hawk H3 nodes and the Hawk H3 API on this GPU, reachable from your Claude or Grok chats through a Cloudflare quick tunnel.

**Before you start**
1. **Runtime → Change runtime type → G4 GPU.** G4 needs a paid Colab plan. Keep a positive compute-unit balance: Colab's free-tier rules forbid web services like this one.
2. **Secrets** (🔑 icon on the left), with *Notebook access* switched on:
   - `ATLAS_API_KEY`: needed for LLM planning.
   - `HAWK_API_TOKEN`: optional; otherwise a new random token is made each session. Make one with `openssl rand -hex 24`.
   - `HF_TOKEN`: optional; for faster or gated Hugging Face downloads.
3. Run the cells in order. Nothing is kept between sessions: each run installs, downloads about 45 GB and starts fresh.

Full guide: [docs/colab.md](https://github.com/Srioff-ashish/Hawk-Minimax-H3-Directory/blob/main/docs/colab.md)

In [ ]:
#@title 1 · Settings { display-mode: "form" }
diffusion_model = "ref2va pruned int8 (21 GB, recommended)"  #@param ["ref2va pruned int8 (21 GB, recommended)", "ref2va pruned fp8 (21 GB)", "ref2va pruned bf16 (40 GB)", "ref2va full int8 (34 GB)", "ref2va full bf16 (66 GB)"]
text_encoder = "nvfp4 (16 GB, recommended on G4)"  #@param ["nvfp4 (16 GB, recommended on G4)", "int8 (27 GB)", "bf16 (52 GB)"]
#@markdown Extra LoRAs, comma-separated: `owner/repo/path/file.safetensors` or a direct download URL. The turbo LoRA is always included.
extra_loras = ""  #@param {type:"string"}
attention = "sol scheduled"  #@param ["sol scheduled", "comfy default"]
install_rife_interpolation = False  #@param {type:"boolean"}
pack_branch = "main"  #@param {type:"string"}

In [ ]:
#@title 2 · Install ComfyUI, Hawk H3 and the API (about 5 minutes)
import importlib, os, subprocess, sys

COMFY_DIR = "/content/ComfyUI"
PACK_DIR = f"{COMFY_DIR}/custom_nodes/Hawk-Minimax-H3-Directory"

if not os.path.isdir(COMFY_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/comfyanonymous/ComfyUI", COMFY_DIR], check=True)
if not os.path.isdir(PACK_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "-b", pack_branch,
                    "https://github.com/Srioff-ashish/Hawk-Minimax-H3-Directory", PACK_DIR], check=True)
else:
    subprocess.run(["git", "-C", PACK_DIR, "pull", "--ff-only"], check=False)

sys.path.insert(0, f"{PACK_DIR}/deploy/colab")
import hawk_colab
importlib.reload(hawk_colab)

hawk_colab.install(COMFY_DIR, PACK_DIR, sol=attention.startswith("sol"), vfi=install_rife_interpolation)

In [ ]:
#@title 3 · Download models (about 45 GB with the defaults)
downloads = hawk_colab.manifest(diffusion_model, text_encoder, extra_loras)
hawk_colab.download_models(COMFY_DIR, downloads, hf_token=hawk_colab.colab_secret("HF_TOKEN"))

In [ ]:
#@title 4 · Start ComfyUI, the tunnel and the API
session = hawk_colab.start(
    COMFY_DIR,
    PACK_DIR,
    diffusion_model=diffusion_model,
    text_encoder=text_encoder,
    attention=attention,
    token=hawk_colab.colab_secret("HAWK_API_TOKEN"),
    atlas_api_key=hawk_colab.colab_secret("ATLAS_API_KEY"),
)

**Connect your chat** with the URLs printed above. The URL is new every session:
- **Claude:** Settings → Connectors → *Add custom connector* → paste the `…/t/<token>/mcp` URL. Remove the previous session's connector first.
- **Grok:** Connectors → New Connector → Custom → the same URL, or `…/mcp` plus the `Authorization: Bearer` header if the form offers one.

Then ask, for example: *"Give me the Hawk H3 upload link"*, *"Plan a 3-segment film from my references"*, *"Render a preview at 0.4 megapixels"*.

Keep cell 5 running while you work. It shows job progress and reopens the tunnel if it drops.

In [ ]:
#@title 5 · Watch jobs (stop this cell any time; the services keep running)
hawk_colab.watch(session, interval=60)

In [ ]:
#@title Show logs (ComfyUI, tunnel, API)
hawk_colab.show_logs(session, lines=60)

In [ ]:
#@title Stop everything
hawk_colab.stop(session)